In [1]:
%pip install -U transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\AMIT\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)
print("DialoGPT chatbot loaded successfully!")

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

C:\Users\AMIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AMIT\.cache\huggingface\hub\models--microsoft--DialoGPT-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  351MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device: cpu
DialoGPT chatbot loaded successfully!


In [4]:
chat_history_ids = None

def chatbot_response(user_input):
    global chat_history_ids

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat(
            [chat_history_ids, new_input_ids],
            dim=-1
        )
    else:
        bot_input_ids = new_input_ids

    # Keep the latest tokens to avoid exceeding model limits
    bot_input_ids = bot_input_ids[:, -900:]

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=100,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
            repetition_penalty=1.2
        )

    response_ids = chat_history_ids[:, bot_input_ids.shape[-1]:]

    response = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True
    )

    return response

In [5]:
user_message = "Hello! How are you?"
response = chatbot_response(user_message)

print("User:", user_message)
print("Chatbot:", response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


User: Hello! How are you?
Chatbot: hi there ! :D i'm new here too , welcome . how is everyone doing ? so far this place has been pretty friendly and fun to be around ... what's up guys amp gals at r bdubsclubs 1.0 .. 2:00 pm EST tomorrow morning Pm edit typo removed lt 3 haha bye friendo XD goodluck on your project brotha ' x3 DDD thanks for the post lolzkatexerpwnzx v2lw7


In [6]:
print("DialoGPT Chatbot")
print("Type 'exit' to end the conversation.\n")

chat_history_ids = None

while True:
    user_message = input("You: ").strip()

    if user_message.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break

    if not user_message:
        print("Chatbot: Please enter a message.")
        continue

    try:
        response = chatbot_response(user_message)
        print("Chatbot:", response if response else "I am not sure how to respond.")
    except Exception as e:
        print("Error generating response:", e)

DialoGPT Chatbot
Type 'exit' to end the conversation.



You:  hlo


Chatbot: nice b8 m80 , would r 8 absolutely hanker 4 that . gl to you too brotha ! lt 3 u 2 tho :D DDD 1v1 me scrub if ur not in the best region on NA LUL xd dnt bully plz fite mi izm boi im good nerd wud 0w it though habbo rofl ty cus kthxbye fran lol bye gracie papi XD srsly is all


You:  Hello! How are you?


Chatbot: hello my name jeff and ill do some testing with this tool for ya thnukneek ? yolo 420blooby tada noicelolahfzzrongenzeyfeelllavrocanadisamameimayyesmylifeupiniogulnoirraaplanfullsurighebeseomeitokareooveryfishyouremorechmecraftpostabwoithccomdeeuvangdoforman


You:  tell me about machine learning


Chatbot: plwhotterbreakoutclonwehiatepfreeverlubutneveronlinecutagainbieforwaberworkrepalohtipprobrolongandmulteehaforknotdustphiamintatbotboygodmathelmodmaxeset420jbytoifronwithmethowbandonpaylessonelovescentasforthdealaiplaythoughwatidtrobblefitsecinkni2getmuchpoinfowhebarallbestwestthisyerregillbr


You:  bye


Chatbot: Goodbye! Have a nice day.


In [7]:
chat_history_ids = None
print("Conversation reset successfully!")

Conversation reset successfully!


In [8]:
print("Experiment completed successfully.")
print("A conversational chatbot was developed using the pre-trained DialoGPT transformer model.")

Experiment completed successfully.
A conversational chatbot was developed using the pre-trained DialoGPT transformer model.
